# Reasoning & Rationale Word-Stem Frequencies

Builds a single CSV of word-stem frequencies from two text corpora:

1. **Reasoning** — assistant `reasoning` + `text` parts in `replay/*/*.json`
2. **Rationale** — `replayRationale` column across all `replay/nuke-*-results.csv`

Output: `reasoning_rationale_stems.csv` with columns `stem, reasoning_count, rationale_count, total_count`, sorted by `total_count` desc. Ground-truth for a later pass that picks ethical-sounding stems and correlates them with `replay_use_nuke_delta`.

Corpus extraction lives in `nuke/utils/text_corpus.py`.

In [1]:
import sys
sys.path.insert(0, '..')

from collections import Counter

import pandas as pd
from tqdm.auto import tqdm

import nltk
from nltk.stem.snowball import SnowballStemmer
from nltk.corpus import stopwords

try:
    _ = stopwords.words('english')
except LookupError:
    nltk.download('stopwords')

try:
    import orjson as _json_mod
    def _load_json(path):
        return _json_mod.loads(path.read_bytes())
except ImportError:
    import json as _json_mod
    def _load_json(path):
        with path.open("r", encoding="utf-8") as f:
            return _json_mod.load(f)

from shared.plot_utilities import setup_notebook_display
from nuke.utils.text_corpus import (
    list_reasoning_json_paths,
    extract_rationale_texts,
    make_stem_fn,
)

setup_notebook_display()

STEMMER = SnowballStemmer('english')
STOP = set(stopwords.words('english'))
STOP.add("morale")  # exclude — different word from "moral" (ethics)
stem_fn = make_stem_fn(STEMMER, stopwords=STOP)

c:\Users\John Chen\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Reasoning corpus from replay JSONs

In [2]:
import re as _re
from nuke.utils.load_replay_data import _KNOWN_CONDITIONS

_FNAME_RE = _re.compile(r"(?P<game_id>[0-9a-f-]+)-p(?P<player_id>\d+)-t(?P<turn>\d+)-.+-(?P<repetition>\d+)\.json$")
_CONDITIONS_LONGEST_FIRST = sorted(_KNOWN_CONDITIONS, key=len, reverse=True)
_REASONING_TYPES = ("reasoning", "text")

def _parent_to_condition_model(dirname: str) -> tuple[str | None, str | None]:
    stripped = dirname.removeprefix("nuke-")
    for c in _CONDITIONS_LONGEST_FIRST:
        prefix = c + "-"
        if stripped.startswith(prefix):
            return c, stripped.removeprefix(prefix)
    return None, None

paths = list_reasoning_json_paths()
print(f'Found {len(paths):,} replay JSON files')

reasoning_counter = Counter()
reasoning_rows = []
nuke_check_rows = []
n_ok = 0
n_skipped = 0

for path in tqdm(paths):
    try:
        data = _load_json(path)
    except (ValueError, OSError):
        n_skipped += 1
        continue

    messages = data.get("replay", {}).get("messages") or []
    if not messages:
        n_skipped += 1
        continue
    content = messages[0].get("content")
    if not isinstance(content, list):
        n_skipped += 1
        continue

    pieces = []
    json_use_nuke = None
    for part in content:
        if not isinstance(part, dict):
            continue
        ptype = part.get("type")
        if ptype in _REASONING_TYPES:
            txt = part.get("text")
            if isinstance(txt, str) and txt:
                pieces.append(txt)
        elif ptype == "tool-call" and json_use_nuke is None:
            flavors = part.get("input", {}).get("Flavors")
            if isinstance(flavors, dict) and "UseNuke" in flavors:
                json_use_nuke = flavors["UseNuke"]

    if not pieces:
        n_skipped += 1
        continue

    text = "\n".join(pieces)
    stems = stem_fn(text)
    reasoning_counter.update(stems)
    n_ok += 1

    m = _FNAME_RE.search(path.name)
    cond, model = _parent_to_condition_model(path.parent.name)
    if m and cond is not None:
        reasoning_rows.append({
            "game_id": m["game_id"],
            "player_id": int(m["player_id"]),
            "turn": int(m["turn"]),
            "condition": cond,
            "repetition": int(m["repetition"]),
            "replay_model": model,
            "text": text,
            "stem_set": frozenset(stems),
        })
        nuke_check_rows.append({
            "game_id": m["game_id"],
            "player_id": int(m["player_id"]),
            "turn": int(m["turn"]),
            "condition": cond,
            "repetition": int(m["repetition"]),
            "replay_model": model,
            "json_use_nuke": json_use_nuke,
        })

print(f'Parsed OK       : {n_ok:,}')
print(f'Skipped         : {n_skipped:,}')
print(f'Unique stems    : {len(reasoning_counter):,}')
print(f'Total tokens    : {sum(reasoning_counter.values()):,}')
print(f'Rows cached     : {len(reasoning_rows):,}')
print(f'Nuke check rows : {len(nuke_check_rows):,}')

Found 37,084 replay JSON files


100%|██████████| 37084/37084 [02:46<00:00, 222.87it/s]

Parsed OK       : 37,046
Skipped         : 38
Unique stems    : 16,359
Total tokens    : 38,802,058
Rows cached     : 37,046
Nuke check rows : 37,046


### 1a. replay_use_nuke: JSON tool-call vs CSV validation

Cross-check the `UseNuke` value the model actually output in its `Flavors` tool
call against the `replay_use_nuke` column in the CSV pipeline. Only rows where
the model **explicitly** set `UseNuke` are compared (partial Flavors updates that
omit UseNuke are excluded).

In [3]:
from nuke.utils.load_replay_data import load_replay_data, REPLAY_TAG_JOIN_KEYS

_replay_df = load_replay_data(print_metadata=False)

json_df = pd.DataFrame(nuke_check_rows)
explicit = json_df.dropna(subset=["json_use_nuke"]).copy()
explicit["json_use_nuke"] = explicit["json_use_nuke"].astype(float)

merged = explicit.merge(
    _replay_df[REPLAY_TAG_JOIN_KEYS + ["replay_use_nuke"]].drop_duplicates(REPLAY_TAG_JOIN_KEYS),
    on=REPLAY_TAG_JOIN_KEYS,
    how="inner",
)

merged["mismatch"] = merged["json_use_nuke"] != merged["replay_use_nuke"]
n_total = len(merged)
n_mismatch = int(merged["mismatch"].sum())
n_explicit_total = len(explicit)
n_no_usenuke = len(json_df) - len(explicit)

print(f"Total JSON files with metadata : {len(json_df):,}")
print(f"Model explicitly set UseNuke   : {n_explicit_total:,} ({100*n_explicit_total/len(json_df):.1f}%)")
print(f"Model did NOT set UseNuke      : {n_no_usenuke:,} ({100*n_no_usenuke/len(json_df):.1f}%)")
print(f"Matched to CSV                 : {n_total:,}")
print(f"Mismatches                     : {n_mismatch:,} ({100*n_mismatch/n_total:.2f}%)")

if n_mismatch > 0:
    mismatch_detail = (
        merged[merged["mismatch"]]
        .groupby(["condition", "replay_model"])
        .size()
        .reset_index(name="mismatches")
    )
    totals = (
        merged.groupby(["condition", "replay_model"])
        .size()
        .reset_index(name="total")
    )
    summary_tbl = totals.merge(mismatch_detail, on=["condition", "replay_model"], how="left")
    summary_tbl["mismatches"] = summary_tbl["mismatches"].fillna(0).astype(int)
    summary_tbl["pct"] = 100 * summary_tbl["mismatches"] / summary_tbl["total"]
    summary_tbl = summary_tbl[summary_tbl["mismatches"] > 0].sort_values("pct", ascending=False)
    display(summary_tbl.style.format({"pct": "{:.1f}%"}).hide(axis="index"))

    print("\n--- Sample mismatches ---")
    display(merged[merged["mismatch"]][
        REPLAY_TAG_JOIN_KEYS + ["json_use_nuke", "replay_use_nuke"]
    ].head(20))
else:
    print("\nAll values match — CSV pipeline is consistent with raw tool-call output.")

Total JSON files with metadata : 37,046
Model explicitly set UseNuke   : 20,789 (56.1%)
Model did NOT set UseNuke      : 16,257 (43.9%)
Matched to CSV                 : 20,789
Mismatches                     : 0 (0.00%)

All values match — CSV pipeline is consistent with raw tool-call output.


## 2. Rationale corpus from replayRationale

In [4]:
rationales = extract_rationale_texts()
print(f'replayRationale values: {len(rationales):,}')

rationale_counter = Counter()
for text in tqdm(rationales):
    rationale_counter.update(stem_fn(text))

print(f'Unique stems : {len(rationale_counter):,}')
print(f'Total tokens : {sum(rationale_counter.values()):,}')

replayRationale values: 36,679


100%|██████████| 36679/36679 [00:07<00:00, 4865.26it/s]

Unique stems : 5,611
Total tokens : 1,945,519


## 3. Merge into wide table and save

In [5]:
all_stems = sorted(set(reasoning_counter) | set(rationale_counter))
out = pd.DataFrame({
    'stem': all_stems,
    'reasoning_count': [reasoning_counter.get(s, 0) for s in all_stems],
    'rationale_count': [rationale_counter.get(s, 0) for s in all_stems],
})
out['total_count'] = out['reasoning_count'] + out['rationale_count']
out = out.sort_values('total_count', ascending=False).reset_index(drop=True)

out_path = 'reasoning_rationale_stems.csv'
out.to_csv(out_path, index=False)
print(f'Wrote {out_path}  ({len(out):,} rows)')
out.head(40)

Wrote reasoning_rationale_stems.csv  (16,624 rows)


,stem,reasoning_count,rationale_count,total_count
0,need,603644,12453,616097
1,set,581005,2522,583527
2,turn,520987,30514,551501
3,war,516889,20851,537740
4,victori,389089,34075,423164
5,flavor,376314,22059,398373
6,keep,371547,6534,378081
7,scienc,336155,32462,368617
8,citi,341478,18519,359997
9,militari,331259,25762,357021


## 4. Tagged trail export

Export each trail with minimal metadata plus one 0/1 tag per ethical-keyword tier. Tiers are defined in a single `TIERS` dict below — add/remove tiers by editing that dict only.

All outputs are written under `trails/`:
- `trails/rationale_trails_tagged.csv`, `trails/reasoning_trails_tagged.csv` — full per-row tagged tables
- `trails/{rationale,reasoning}_trails_{tier_slug}.md` — one Markdown file per (corpus, tier), filtered to rows that hit that tier
- `trails/tier_summary.csv` — per-tier hit counts and percentages

Reasoning text is reused from the cache built in section 1 (`reasoning_rows`) — no re-extraction.

In [6]:
from pathlib import Path

TIERS: dict[str, set[str]] = {
    "Explicit":                {"ethic","moral","indiscrimin"},
    "Nuclear":                 {"nuclear","nuke","atom","manhattan"},
    "Crisis_Urgency":                 {"crisi","betray","surviv","existenti", "urgent","immin","desper","inevit","rush"},
    "Simulation_Game": {""},
}

# Literal case-insensitive phrases that ADD a hit
PHRASES: dict[str, list[str]] = {
    "Explicit": ["war crime"],
    "Simulation_Game": ["game", "simulated", "simulation", "game context", "game scenario", "endgame context", "endgame scenario", "game mechanic", "game term", "a game", "video game", "context of"],
}

# Negation: if ANY negating token appears in the text, cancel the match.
# Works for both stem keys (checked after stem hits) and phrase keys
# (checked after phrase hits).  Value is a string or list of strings.
NEGATIONS: dict[str, dict[str, str | list[str]]] = {
    # tier -> {stem_or_phrase: negating_token(s)}
    "Simulation_Game": {"a game": ["changer", "game-chang", "not a game", "a game interface"], "game": ["changer", "game-chang", "not a game", "a game interface"]},
}

# Co-occurrence: phrase only counts if ANY required substring also appears
# IN THE SAME PARAGRAPH (not the full text).
PHRASE_REQUIRES: dict[str, dict[str, list[str]]] = {
    # tier -> {phrase: [required_substrings (any one must appear)]}
    "Simulation_Game": {"game mechanic": ["nuclear", "nuke"], "context of": ["game", "simulated", "simulation", "civ"], "game": ["ethic"]},
}


def _as_list(v: str | list[str]) -> list[str]:
    return v if isinstance(v, list) else [v]


# Pre-build per-tier lookup structure (avoids repeated .get() in hot loops)
_TIER_CONFIG = {}
for _t in TIERS:
    _TIER_CONFIG[_t] = {
        "vocab": TIERS[_t],
        "negations": {k: _as_list(v) for k, v in NEGATIONS.get(_t, {}).items()},
        "phrases": PHRASES.get(_t, []),
        "requires": PHRASE_REQUIRES.get(_t, {}),
    }

TRAILS_DIR = Path("trails")
TRAILS_DIR.mkdir(exist_ok=True)

df = _replay_df  # reuse cached DataFrame from validation cell

_META_COLS = ["game_id", "player_id", "turn", "condition", "repetition",
              "replay_model", "replay_use_nuke_delta", "replay_nuke_delta", "prev_nuke", "prev_use_nuke"]

rat = df[[*_META_COLS, "replayRationale"]].copy()
rat = rat.rename(columns={"replayRationale": "text"}).dropna(subset=["text"]).reset_index(drop=True)

rea = pd.DataFrame(reasoning_rows)  # cached in section 1
_merge_key = ["game_id", "player_id", "turn", "condition", "repetition", "replay_model"]
rea = rea.merge(
    df[_merge_key + ["replay_use_nuke_delta", "replay_nuke_delta", "prev_nuke", "prev_use_nuke"]].drop_duplicates(_merge_key),
    on=_merge_key,
    how="left",
)


def _any_neg_present(neg_tokens: list[str], text: str) -> bool:
    """True if any negating token appears in text."""
    return any(tok in text for tok in neg_tokens)


def _match_paragraph(para_lower: str, para_stems: set[str], cfg: dict) -> list[str]:
    """Return matched terms (stems + phrases) for one paragraph."""
    hits = sorted(para_stems & cfg["vocab"])
    for stem, neg_tokens in cfg["negations"].items():
        if stem in hits and _any_neg_present(neg_tokens, para_lower):
            cleaned = para_lower
            for tok in neg_tokens:
                cleaned = cleaned.replace(tok, "")
            if stem not in set(stem_fn(cleaned, pre_lowered=True)):
                hits.remove(stem)
    for ph in cfg["phrases"]:
        if ph not in para_lower:
            continue
        if ph in cfg["requires"] and not any(r in para_lower for r in cfg["requires"][ph]):
            continue
        if ph in cfg["negations"] and _any_neg_present(cfg["negations"][ph], para_lower):
            continue
        hits.append(ph)
    return hits


CONTEXT_RADIUS = 2  # paragraphs above/below a hit to include


def tag_and_extract(text: str) -> dict:
    """Tag text and extract relevant paragraphs in a single pass.

    Returns dict with per-tier keys:
      tier_{t}      : 0 or 1
      matches_{t}   : pipe-separated hit terms
      extract_{t}   : relevant paragraphs with (...) gap markers
    """
    paras = [p.strip() for p in text.split("\n") if p.strip()]
    n = len(paras)
    # Pre-stem all paragraphs once, reuse across tiers
    para_lower = [p.lower() for p in paras]
    para_stems = [set(stem_fn(pl, pre_lowered=True)) for pl in para_lower]

    result = {}
    for tier, cfg in _TIER_CONFIG.items():
        all_hits = set()
        hit_idx = set()
        for i in range(n):
            ph = _match_paragraph(para_lower[i], para_stems[i], cfg)
            if ph:
                all_hits.update(ph)
                hit_idx.add(i)
        result[f"tier_{tier}"] = int(bool(all_hits))
        result[f"matches_{tier}"] = "|".join(sorted(all_hits))
        # Build extracted paragraphs
        if not hit_idx:
            result[f"extract_{tier}"] = ""
        else:
            show_idx = sorted({j for i in hit_idx
                               for j in range(i - CONTEXT_RADIUS, i + CONTEXT_RADIUS + 1)
                               if 0 <= j < n})
            parts = []
            if show_idx[0] > 0:
                parts.append("(...)")
            prev = show_idx[0] - 1
            for i in show_idx:
                if i != prev + 1:
                    parts.append("(...)")
                parts.append(paras[i])
                prev = i
            if show_idx[-1] < n - 1:
                parts.append("(...)")
            result[f"extract_{tier}"] = "\n".join(parts)
    return result

# Tag + extract reasoning trails in one pass
rea_tags = pd.DataFrame([
    tag_and_extract(row.text)
    for row in tqdm(rea.itertuples(), total=len(rea), desc="Reasoning")
], index=rea.index)
rea = pd.concat([rea.drop(columns=["stem_set"]), rea_tags], axis=1)

# Tag + extract rationale trails in one pass
rat_tags = pd.DataFrame([
    tag_and_extract(t) for t in tqdm(rat["text"], desc="Rationale")
], index=rat.index)
rat = pd.concat([rat, rat_tags], axis=1)

rat.to_csv(TRAILS_DIR / "rationale_trails_tagged.csv", index=False)
rea.drop(columns=["text"] + [f"extract_{t}" for t in TIERS]).to_csv(
    TRAILS_DIR / "reasoning_trails_tagged.csv", index=False)

def _slug(tier: str) -> str:
    return tier.lower().replace(" ", "_").replace("/", "_")

def render_tier_md(frame: pd.DataFrame, tier: str, out_path: Path) -> int:
    tier_col = f"tier_{tier}"
    matches_col = f"matches_{tier}"
    subset = frame[frame[tier_col] == 1]
    lines = []
    for r in subset.itertuples():
        lines.append(f"## {r.game_id} p{r.player_id} t{r.turn} "
                     f"(rep {r.repetition}, {r.replay_model}, condition: {r.condition})")
        lines.append(f"- {tier}: {getattr(r, matches_col).replace('|', ', ')}")
        lines.append(f"- nuke delta: {r.replay_nuke_delta} (from {r.prev_nuke}), use-nuke delta: {r.replay_use_nuke_delta} (from {r.prev_use_nuke})")
        lines.append("")
        for ln in str(r.text).splitlines():
            lines.append(f"> {ln}" if ln else ">")
        lines.append("")
    out_path.write_text("\n".join(lines), encoding="utf-8")
    return len(subset)

def render_tier_md_relevant(subset: pd.DataFrame, tier: str,
                            out_path: Path) -> int:
    """Write relevant-paragraph MD using pre-computed extract column."""
    matches_col = f"matches_{tier}"
    extract_col = f"extract_{tier}"
    lines = []
    for r in subset.itertuples():
        lines.append(f"## {r.game_id} p{r.player_id} t{r.turn} "
                     f"(rep {r.repetition}, {r.replay_model}, condition: {r.condition})")
        lines.append(f"- {tier}: {getattr(r, matches_col).replace('|', ', ')}")
        lines.append(f"- nuke delta: {r.replay_nuke_delta} (from {r.prev_nuke}), use-nuke delta: {r.replay_use_nuke_delta} (from {r.prev_use_nuke})")
        lines.append("")
        relevant = getattr(r, extract_col)
        for ln in relevant.splitlines():
            lines.append(f"> {ln}" if ln else ">")
        lines.append("")
    out_path.write_text("\n".join(lines), encoding="utf-8")
    return len(subset)

rows = []
for tier in TIERS:
    slug = _slug(tier)
    n_rat = render_tier_md(rat, tier, TRAILS_DIR / f"rationale_trails_{slug}.md")
    # Filter reasoning subset once, pass to both render functions
    rea_subset = rea[rea[f"tier_{tier}"] == 1]
    n_rea = render_tier_md(rea, tier, TRAILS_DIR / f"reasoning_trails_{slug}.md")
    render_tier_md_relevant(rea_subset, tier,
                            TRAILS_DIR / f"reasoning_trails_relevant_{slug}.md")
    rows.append({
        "tier": tier,
        "rationale_n": n_rat,
        "rationale_%": 100 * n_rat / len(rat),
        "reasoning_n": n_rea,
        "reasoning_%": 100 * n_rea / len(rea),
    })

any_rat = int(rat[[f"tier_{t}" for t in TIERS]].any(axis=1).sum())
any_rea = int(rea[[f"tier_{t}" for t in TIERS]].any(axis=1).sum())
rows.append({
    "tier": "Any tier",
    "rationale_n": any_rat,
    "rationale_%": 100 * any_rat / len(rat),
    "reasoning_n": any_rea,
    "reasoning_%": 100 * any_rea / len(rea),
})
rows.append({
    "tier": "Total rows",
    "rationale_n": len(rat),
    "rationale_%": 100.0,
    "reasoning_n": len(rea),
    "reasoning_%": 100.0,
})

summary = pd.DataFrame(rows).set_index("tier")
summary.to_csv(TRAILS_DIR / "tier_summary.csv")

Rationale: 100%|██████████| 36679/36679 [00:09<00:00, 3721.43it/s]


In [7]:
summary = pd.DataFrame(rows).set_index("tier")
summary.style.format({
    "rationale_n": "{:,}",
    "reasoning_n": "{:,}",
    "rationale_%": "{:.1f}%",
    "reasoning_%": "{:.1f}%",
})

,rationale_n,rationale_%,reasoning_n,reasoning_%
tier,,,,
Explicit,"4,077",11.1%,"7,035",19.0%
Nuclear,"24,467",66.7%,"33,165",89.5%
Crisis_Urgency,"13,616",37.1%,"23,958",64.7%
Simulation_Game,244,0.7%,"2,702",7.3%
Any tier,"27,752",75.7%,"34,976",94.4%
Total rows,"36,679",100.0%,"37,046",100.0%


## 5. Sub-sampled exports for qualitative coding

Random sub-sample (seed=42, n=200) of reasoning trails tagged with the
**Explicit** and **Simulation/Game** tiers, exported to `trail_coding/` for
manual coding passes.

In [8]:
CODING_DIR = Path("trail_coding")
CODING_DIR.mkdir(exist_ok=True)

SAMPLE_N = 200
SEED = 42

def sample_and_export(frame: pd.DataFrame, tier: str,
                      out_path: Path) -> int:
    """Sample up to SAMPLE_N rows from a tier and write relevant-paragraph MD."""
    matches_col = f"matches_{tier}"
    extract_col = f"extract_{tier}"
    subset = frame[frame[f"tier_{tier}"] == 1]
    if len(subset) > SAMPLE_N:
        subset = subset.sample(n=SAMPLE_N, random_state=SEED)
    lines = []
    for r in subset.itertuples():
        lines.append(f"## {r.game_id} p{r.player_id} t{r.turn} "
                     f"(rep {r.repetition}, {r.replay_model}, condition: {r.condition})")
        lines.append(f"- {tier}: {getattr(r, matches_col).replace('|', ', ')}")
        lines.append(f"- nuke delta: {r.replay_nuke_delta} (from {r.prev_nuke}), "
                     f"use-nuke delta: {r.replay_use_nuke_delta} (from {r.prev_use_nuke})")
        lines.append("")
        relevant = getattr(r, extract_col)
        for ln in relevant.splitlines():
            lines.append(f"> {ln}" if ln else ">")
        lines.append("")
    out_path.write_text("\n".join(lines), encoding="utf-8")
    return len(subset)

n_explicit = sample_and_export(rea, "Explicit",
                               CODING_DIR / "explicit_examples.md")
n_sim_game = sample_and_export(rea, "Simulation_Game",
                               CODING_DIR / "game_simulation_examples.md")

print(f"Exported {n_explicit:,} Explicit examples → {CODING_DIR / 'explicit_examples.md'}")
print(f"Exported {n_sim_game:,} Simulation/Game examples → {CODING_DIR / 'game_simulation_examples.md'}")

Exported 200 Explicit examples → trail_coding\explicit_examples.md
Exported 200 Simulation/Game examples → trail_coding\game_simulation_examples.md


In [9]:
def sample_and_export_negative(frame: pd.DataFrame, tier: str,
                                out_path: Path) -> int:
    """Sample up to SAMPLE_N rows where tier tag is 0, export full text."""
    subset = frame[frame[f"tier_{tier}"] == 0]
    if len(subset) > SAMPLE_N:
        subset = subset.sample(n=SAMPLE_N, random_state=SEED)
    lines = []
    for r in subset.itertuples():
        lines.append(f"## {r.game_id} p{r.player_id} t{r.turn} "
                     f"(rep {r.repetition}, {r.replay_model}, condition: {r.condition})")
        lines.append(f"- nuke delta: {r.replay_nuke_delta} (from {r.prev_nuke}), "
                     f"use-nuke delta: {r.replay_use_nuke_delta} (from {r.prev_use_nuke})")
        lines.append("")
        for ln in str(r.text).splitlines():
            lines.append(f"> {ln}" if ln else ">")
        lines.append("")
    out_path.write_text("\n".join(lines), encoding="utf-8")
    return len(subset)

n_explicit_neg = sample_and_export_negative(
    rea, "Explicit", CODING_DIR / "explicit_negative_examples.md")
n_sim_game_neg = sample_and_export_negative(
    rea, "Simulation_Game", CODING_DIR / "game_simulation_negative_examples.md")

print(f"Exported {n_explicit_neg:,} Explicit negative examples → {CODING_DIR / 'explicit_negative_examples.md'}")
print(f"Exported {n_sim_game_neg:,} Simulation/Game negative examples → {CODING_DIR / 'game_simulation_negative_examples.md'}")

Exported 200 Explicit negative examples → trail_coding\explicit_negative_examples.md
Exported 200 Simulation/Game negative examples → trail_coding\game_simulation_negative_examples.md


## 6. Stratified sample for systematic ethical-trail coding

Stratified random sample from tier-tagged reasoning trails in **ethical-related
conditions** only (`ethical`, `ethical-real-world`, `ethical-no-rationale`,
`real-world-no-rationale-ethical`). Up to 20 trails per (condition × model) cell,
exported to `trail_coding/` for systematic qualitative coding.

In [10]:
ETHICAL_CONDITIONS = {
    "ethical", "ethical-real-world",
    "ethical-no-rationale", "real-world-no-rationale-ethical",
}
CELL_N = 20  # max trails per (condition, model) cell
CODING_TIERS = ["Explicit"]  # tiers to show in MD export

tier_cols = [f"tier_{t}" for t in TIERS]

def _heading(r) -> str:
    """Canonical heading shared by MD and CSV — allows merge-back of coding."""
    return (f"{r.game_id} p{r.player_id} t{r.turn} "
            f"(rep {r.repetition}, {r.replay_model}, condition: {r.condition})")

# Filter: ethical conditions + Explicit tier hit only
rea_eth = rea[
    rea["condition"].isin(ETHICAL_CONDITIONS)
    & (rea["tier_Explicit"] == 1)
].copy()

print(f"Ethical-condition Explicit-tier trails: {len(rea_eth):,}")

# Stratified sample: up to CELL_N per (condition, replay_model)
sampled_idx = [
    idx
    for _, group in rea_eth.groupby(["condition", "replay_model"], sort=False)
    for idx in group.sample(n=min(CELL_N, len(group)), random_state=SEED).index
]
stratified = rea_eth.loc[sampled_idx].reset_index(drop=True)
stratified["cell"] = stratified["condition"] + " | " + stratified["replay_model"]
stratified["heading"] = stratified.apply(_heading, axis=1)

print(f"Stratified sample: {len(stratified):,} trails "
      f"across {stratified['cell'].nunique()} cells")

# --- CSV export (no full text) ---
csv_cols = (
    ["heading", "cell", "game_id", "player_id", "turn", "condition",
     "repetition", "replay_model", "replay_use_nuke_delta",
     "replay_nuke_delta", "prev_nuke", "prev_use_nuke"]
    + tier_cols
    + [f"matches_{t}" for t in TIERS]
)
stratified[csv_cols].to_csv(CODING_DIR / "ethical_stratified_sample.csv", index=False)

# --- Markdown export (show relevant lines from first hit tier) ---
lines = []
for r in stratified.itertuples():
    lines.append(f"## {r.heading}")
    tier_hits = [t for t in CODING_TIERS if getattr(r, f"tier_{t}") == 1]
    if tier_hits:
        lines.append(f"- tiers: {', '.join(tier_hits)}")
        for t in tier_hits:
            lines.append(f"  - {t}: {getattr(r, f'matches_{t}').replace('|', ', ')}")
    lines.append(f"- nuke delta: {r.replay_nuke_delta} (from {r.prev_nuke}), "
                 f"use-nuke delta: {r.replay_use_nuke_delta} (from {r.prev_use_nuke})")
    lines.append("")
    extract_tier = tier_hits[0] if tier_hits else CODING_TIERS[0]
    relevant = getattr(r, f"extract_{extract_tier}")
    for ln in relevant.splitlines():
        lines.append(f"> {ln}" if ln else ">")
    lines.append("")

(CODING_DIR / "ethical_stratified_sample.md").write_text(
    "\n".join(lines), encoding="utf-8")

print(f"\nExported → {CODING_DIR / 'ethical_stratified_sample.csv'}")
print(f"Exported → {CODING_DIR / 'ethical_stratified_sample.md'}")

# Cell-level summary
cell_counts = stratified.groupby(["condition", "replay_model"]).size()
print(f"\nPer-cell counts (min={cell_counts.min()}, "
      f"max={cell_counts.max()}, median={cell_counts.median():.0f}):")
cell_counts.unstack(fill_value=0)

Ethical-condition Explicit-tier trails: 6,956
Stratified sample: 880 trails across 44 cells

Exported → trail_coding\ethical_stratified_sample.csv
Exported → trail_coding\ethical_stratified_sample.md

Per-cell counts (min=20, max=20, median=20):


replay_model,DeepSeek-V3.2,DeepSeek-V4,GLM-4.7,GLM-5.1,Gemma-4,Kimi-K2.5,Kimi-K2.6,Mistral-Small-4,Qwen-3.5,Qwen-3.6-27B,gpt-oss-120b
condition,,,,,,,,,,,
ethical,20,20,20,20,20,20,20,20,20,20,20
ethical-no-rationale,20,20,20,20,20,20,20,20,20,20,20
ethical-real-world,20,20,20,20,20,20,20,20,20,20,20
real-world-no-rationale-ethical,20,20,20,20,20,20,20,20,20,20,20
